# Correctness and Invariants


```{contents}
:local:
:depth: 2
```

A fast expense report is useful only if its totals and selected records satisfy the business rules. Specify the result first, then connect every loop step to that specification.

## Specifications and Failure Policies

```{index} Specifications and Failure Policies
```


For a maximum, require a non-null, nonempty array. Promise that the result is an input value and is at least every input value. Merely promising a large value is insufficient: returning int.MaxValue would satisfy that weaker promise. Define rejected inputs and whether failure changes state.

In [ ]:
using System;
using System.Linq;

foreach (int[] values in new[] { new[] { -9, -2, -7 }, new[] { 5 }, Array.Empty<int>() })
{
    try { Console.WriteLine($"maximum={MaxExpense(values)}"); }
    catch (ArgumentException) { Console.WriteLine("empty input rejected"); }
}
long MaxExpense(int[] values)
{
    if (values is null) throw new ArgumentNullException(nameof(values));
    if (values.Length == 0) throw new ArgumentException("At least one expense is required.");
    int best = values[0];
    for (int i = 1; i < values.Length; i++)
        if (values[i] > best) best = values[i];
    return best;
}

Runtime guards enforce this contract. Other algorithms may document a precondition rather than check it each time. Binary search requires sorted input; checking that condition costs Θ(n) and changes the cost of an individual search.

## An Invariant with an Exact Boundary

```{index} An Invariant with an Exact Boundary
```


At the head of the maximum loop, `1 ≤ i ≤ n` and `best` is the maximum of the processed prefix `[0, i)`. Half-open notation includes index zero and excludes index `i`; it avoids uncertainty about whether the next value has already been processed.

In [ ]:
using System;
using System.Linq;

int[] values = { -9, -2, -7, -2 };
int best = values[0];
for (int i = 1; i < values.Length; i++)
{
    Console.WriteLine($"before i={i}: processed={i}, best={best}");
    if (values[i] > best) best = values[i];
    Console.WriteLine($"after i={i}: processed={i+1}, best={best}");
}
Console.WriteLine($"maximum={best}");

Initialization: at `i=1`, the prefix contains only `values[0]`. Maintenance: comparing the next value extends the prefix by one while preserving its maximum. At normal loop exit `i=n`, the prefix is the entire input, so the invariant gives both parts of the postcondition.

## Termination and Total Correctness

```{index} Termination and Total Correctness
```


An invariant explains the result if execution finishes: partial correctness. Total correctness also requires termination. A variant is a nonnegative integer measure that strictly decreases on each continuing iteration. For the maximum loop, `n−i` decreases by one.

In [ ]:
using System;
using System.Linq;

int remaining = 3;
while (remaining > 0)
{
    Console.WriteLine($"variant before={remaining}");
    remaining--;
}
Console.WriteLine($"variant at exit={remaining}");

An unchanged `remaining` could preserve a safety invariant while looping forever. The decreasing measure rules out that behavior. The third step of an invariant argument uses the exit condition to derive the result; it does not by itself prove that the exit will eventually occur.

## Early Return and First-Match Search

```{index} Early Return and First-Match Search
```


Specify first-match search: return the smallest matching index, or −1 when no match exists. Before testing index `i`, every index in `[0,i)` fails to match. Normal loop exit and early return require separate arguments.

In [ ]:
using System;
using System.Linq;

int[] values = { 5, 8, 8, 14 };
foreach (int target in new[] { 8, 14, 99 })
    Console.WriteLine($"target={target}, first index={FirstIndex(values,target)}");
int FirstIndex(int[] values, int target)
{
    for (int i = 0; i < values.Length; i++)
        if (values[i] == target) return i;
    return -1;
}

An early return identifies a matching value and the invariant excludes earlier matches. At normal exit, the invariant excludes every index. Empty input returns −1 without an array access. The variant `n−i` proves the continuing loop terminates.

## Boundary Tests and a Counterexample

```{index} Boundary Tests and a Counterexample
```


Tests complement proofs by challenging contracts and implementation details. A common maximum bug initializes best to zero; the bug stays hidden on positive data. Test singleton, negative, duplicate, extreme, empty, and rejected inputs.

In [ ]:
using System;
using System.Linq;

int[][] cases = { new[] { 0 }, new[] { -5,-2,-9 }, new[] { 7,7 }, new[] { int.MinValue,int.MaxValue } };
foreach (int[] values in cases)
{
    long actual = MaxExpense(values);
    if (actual != values.Max()) throw new Exception("Maximum mismatch.");
    Console.WriteLine($"[{string.Join(",",values)}] -> {actual}");
}
int broken = 0;
foreach (int value in new[] { -5,-2,-9 }) if (value > broken) broken = value;
Console.WriteLine($"zero-initialization counterexample: got={broken}, expected=-2");
long MaxExpense(int[] values)
{
    if (values is null) throw new ArgumentNullException(nameof(values));
    if (values.Length == 0) throw new ArgumentException("At least one expense is required.");
    int best = values[0];
    for (int i = 1; i < values.Length; i++)
        if (values[i] > best) best = values[i];
    return best;
}

The reference comparison checks actual code on these inputs. The invariant covers every input satisfying the contract. Arithmetic behavior is part of the specification too: use a sufficiently wide type or a checked operation when a mathematical sum may overflow the chosen C# type.

## Exercise

```{index} Analysis practice
```

Count nonnegative adjustments. State the processed-prefix invariant and variant; include zero as nonnegative.

In [ ]:
using System;

// Exercise
// Count nonnegative adjustments. State the processed-prefix invariant and variant; include zero as nonnegative.
// Your code starts here.
/* TODO */
// Your code ends here.

In [1]:
using System;

// Solution
int[] values={-3,0,5,-1};
int count=0;
// At loop head, count is the number of nonnegative entries in [0,i).
// Variant is values.Length-i.
for(int i=0;i<values.Length;i++)if(values[i]>=0)count++;
Console.WriteLine($"nonnegative adjustments={count}");

nonnegative adjustments=2


```{rubric} Footnotes
```
[^1]: A specification must identify the allowed inputs, promised result, and failure behavior; a proof is only as useful as that specification.
[^2]: Testing a finite sample generally cannot prove a statement over all possible input sizes. Boundary tests and counterexamples remain valuable implementation checks.